In [1]:
!pip install selenium


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

all_periods = []

driver.get("https://egymonuments.gov.eg/en/historical-periods")
# Waiting for the collection links to load
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-timeline a")))
i = 1
while True:
    try:
        # press the next site link
        try:
            site_link = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f'//*[@id="cd-timeline"]/inners-period-listing/div[{i}]/a/div[2]/a')
            ))
            site_link.click()
            i += 1
        except:
            break

        # collect the data
        period_name = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.mainPageTitle'))).text
        period_from_to = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont.periodFromTo > div.dateFromToCont'
        ).text
        period_description = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont.periodFromTo > div.DescriptionSection > div.periodTxtCont > div.txtSection'
        ).text
        try:
            img_elem = driver.find_element(By.CSS_SELECTOR,
                'div.relative.descriptionImageCont.periodFromTo > div.DescriptionSection > div.periodImgCont > img'
            )
            place_photo = img_elem.get_attribute("src")
        except:
            place_photo = ""
        all_periods.append({
            "collection": period_name,
            "location": period_from_to,
            "Description": period_description,
            "photo_url": place_photo
        })

        # previous page
        driver.back()
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-timeline a")))


    except Exception as e:
        print("Finished — no more items.")
        break

print(f"Total periods: {len(all_periods)}")

Total periods: 23


In [10]:
df = pd.DataFrame(all_periods)
df

,collection,location,Description,photo_url
0,Predynastic Period,5500 BC\n/\n3100 BC,This period covers all of ancient Egyptian pre...,https://egymonuments.gov.eg/media/1344/pre-dyn...
1,Early Dynastic Period,3100 BC\n/\n2686 BC,Ancient Egyptian history before the Graeco‑Rom...,https://egymonuments.gov.eg/media/1010/pre.jpg...
2,Old Kingdom,2686 BC\n/\n2181 BC,The Third to Sixth Dynasties make up the Old K...,https://egymonuments.gov.eg/media/1021/old-kin...
3,First Intermediate Period,2181 BC\n/\n2055 BC,The weakness of the rulers of the Sixth Dynast...,https://egymonuments.gov.eg/media/1035/first-i...
4,Middle Kingdom,2055 BC\n/\n1650 BC,The Middle Kingdom was a time of great prosper...,https://egymonuments.gov.eg/media/1067/4a392f4...
5,Second Intermediate Period,1650 BC\n/\n1550 BC,Already during the Twelfth Dynasty in the Midd...,https://egymonuments.gov.eg/media/1072/d17.jpg...
6,New Kingdom,1550 BC\n/\n1069 BC,"During the New Kingdom, Egypt became a great e...",https://egymonuments.gov.eg/media/1083/new-kin...
7,Third Intermediate Period,1069 BC\n/\n747 BC,"The Third Intermediate Period was, on the whol...",https://egymonuments.gov.eg/media/1092/third2....
8,Late Period,747 BC\n/\n332 BC,"When the Kushite kings conquered Egypt, one of...",https://egymonuments.gov.eg/media/1508/late-pe...
9,Ptolemaic Period,332 BC\n/\n30 BC,After Alexander the Great defeated the Persian...,https://egymonuments.gov.eg/media/1243/ptolema...


In [11]:
#change name of location column to from_to
df.rename(columns={"location": "from_to"}, inplace=True)

In [12]:
df.to_csv('periods_en.csv', index=False)